In [1]:
import html, json, re
from pathlib import Path
import pandas as pd

def strip_html(s):
    s = html.unescape(s or "")          # Greenhouse double-escapes its HTML
    s = re.sub(r"<[^>]+>", " ", s)
    s = html.unescape(s)
    return re.sub(r"\s+", " ", s).strip()

EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE = re.compile(r"\(?\b\d{3}\)?[-.\s]\d{3}[-.\s]\d{4}\b")

def scrub(s):
    return PHONE.sub("[phone]", EMAIL.sub("[email]", s))

In [2]:
rows = []
for path in sorted(Path("data/raw").glob("*/*.json")):
    snapshot = path.parent.name
    ats, board = path.stem.split("_", 1)
    data = json.loads(path.read_text())

    if ats == "greenhouse":
        for j in data["jobs"]:
            rows.append({
                "job_id": f"gh_{j['id']}",
                "company": j.get("company_name") or board,
                "ats": ats,
                "title": j["title"],
                "location": (j.get("location") or {}).get("name"),
                "posted_at": j.get("first_published"),
                "description": strip_html(j.get("content")),
                "url": j.get("absolute_url"),
                "snapshot": snapshot,
            })
    else:  # lever returns a bare list
        for j in data:
            lists = " ".join(f"{l['text']} {l['content']}" for l in j.get("lists", []))
            text = " ".join([j.get("descriptionPlain", ""), strip_html(lists), j.get("additionalPlain", "")])
            rows.append({
                "job_id": f"lv_{j['id']}",
                "company": board,
                "ats": ats,
                "title": j["text"],
                "location": (j.get("categories") or {}).get("location"),
                "posted_at": pd.to_datetime(j.get("createdAt"), unit="ms", utc=True).isoformat(),
                "description": re.sub(r"\s+", " ", text).strip(),
                "url": j.get("hostedUrl"),
                "snapshot": snapshot,
            })

raw = pd.DataFrame(rows)
print(raw.shape)
raw.head()

(4620, 9)


,job_id,company,ats,title,location,posted_at,description,url,snapshot
0,gh_8039471003,Affirm,greenhouse,"Analyst, Early Career (Full Stack)","New York, New York, United States; San Francis...",2026-10-21T17:16:19-04:00,"At Affirm, we exist for the moments that matte...",https://job-boards.greenhouse.io/affirm/jobs/8...,2026-11-08
1,gh_7832318003,Affirm,greenhouse,"Analyst II, Credit",Remote UK,2026-09-07T10:40:17-04:00,"At Affirm, we exist for the moments that matte...",https://job-boards.greenhouse.io/affirm/jobs/7...,2026-11-08
2,gh_7916080003,Affirm,greenhouse,"Analytics Engineer II, Full Stack (Revenue Ana...",Remote Spain,2026-08-20T09:43:45-04:00,"At Affirm, we exist for the moments that matte...",https://job-boards.greenhouse.io/affirm/jobs/7...,2026-11-08
3,gh_7813362003,Affirm,greenhouse,"Analytics Engineer II, Full Stack (Revenue Ana...",Remote Poland,2026-07-28T06:52:21-04:00,"At Affirm, we exist for the moments that matte...",https://job-boards.greenhouse.io/affirm/jobs/7...,2026-11-08
4,gh_8004096003,Affirm,greenhouse,"Analytics Lead, Deposit Fraud Risk",Remote US,2026-09-29T13:22:14-04:00,"At Affirm, we exist for the moments that matte...",https://job-boards.greenhouse.io/affirm/jobs/8...,2026-11-08


In [3]:
ROLES = {   # order matters: first match wins
    "ai_ml": r"machine learning|\bml\b|\bai engineer|applied (ai|scientist)|\bllm\b|deep learning|computer vision|\bnlp\b|mlops",
    "data_science": r"data scien",
    "analyst": r"data analy|analytics|business intelligence|\bbi analyst",
}

def classify(title):
    t = title.lower()
    for role, pat in ROLES.items():
        if re.search(pat, t):
            return role
    return "other"

df = raw.copy()
df["description"] = df["description"].map(scrub)
df["role"] = df["title"].map(classify)
df["posted_at"] = pd.to_datetime(df["posted_at"], utc=True, format="ISO8601").dt.date
df["first_seen"] = df.groupby("job_id")["snapshot"].transform("min")
df["last_seen"] = df.groupby("job_id")["snapshot"].transform("max")
df["n_snapshots"] = df.groupby("job_id")["snapshot"].transform("nunique")
df = df.sort_values("snapshot").drop_duplicates("job_id", keep="last").drop(columns="snapshot")
df = df[df["description"].str.len() > 0]

print(df.shape)
print(df["role"].value_counts())
df.isna().sum()

(4620, 12)
role
other           4353
ai_ml            142
data_science      71
analyst           54
Name: count, dtype: int64


job_id         0
company        0
ats            0
title          0
location       0
posted_at      0
description    0
url            0
role           0
first_seen     0
last_seen      0
n_snapshots    0
dtype: int64

In [4]:
Path("data/clean").mkdir(exist_ok=True)
df.to_csv("data/clean/postings.csv", index=False)

sample = df[df["role"] != "other"].sample(20, random_state=1).copy()
sample["description"] = sample["description"].str[:300] + "..."
sample.to_csv("data/sample_postings.csv", index=False)
sample.head()

,job_id,company,ats,title,location,posted_at,description,url,role,first_seen,last_seen,n_snapshots
3049,gh_8203478,Reddit,greenhouse,"Staff Machine Learning Engineer, AI Security",Remote - United States,2026-09-15,Reddit is a community of communities. It’s bui...,https://job-boards.greenhouse.io/reddit/jobs/8...,ai_ml,2026-11-08,2026-11-08,1
2977,gh_8355688,Reddit,greenhouse,Senior Data Scientist,"San Francisco, CA",2026-11-05,Reddit is a community of communities. It’s bui...,https://job-boards.greenhouse.io/reddit/jobs/8...,data_science,2026-11-08,2026-11-08,1
2845,gh_5740948,Pinterest,greenhouse,"Sr. Data Scientist, Monetization","San Francisco, CA, US; Seattle, WA, US",2024-02-22,About Pinterest: Millions of people around the...,https://www.pinterestcareers.com/jobs/?gh_jid=...,data_science,2026-11-08,2026-11-08,1
2532,gh_8221941,Omada Health,greenhouse,"Senior Director, Capacity Planning, Workforce ...","Remote, USA",2026-09-24,Omada Health is on a mission to bend the curve...,https://job-boards.greenhouse.io/omadahealth/j...,analyst,2026-11-08,2026-11-08,1
3441,gh_8224915,Stripe,greenhouse,"Machine Learning Engineer, Growth Platform",San Francisco,2026-09-28,Who we are About Stripe Stripe is a financial ...,https://stripe.com/jobs/search?gh_jid=8224915,ai_ml,2026-11-08,2026-11-08,1
